## Homework-03

In [159]:
import numpy as np 
import pandas as pd

In [160]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'

In [161]:
!curl $data > course_lead_scoring.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80876  100 80876    0     0   190k      0 --:--:-- --:--:-- --:--:--  191k


In [162]:
df = pd.read_csv('course_lead_scoring.csv')

In [163]:
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [164]:
categorical_columns =  list(df.dtypes[df.dtypes == 'object'].index)
categorical_columns

['lead_source', 'industry', 'employment_status', 'location']

In [165]:
for c in categorical_columns:
    df[c] = df[c].fillna('NA')

In [166]:
numerical = list(df.dtypes[(df.dtypes == 'int64' ) |  (df.dtypes == 'float64')].index)
numerical.pop(-1)
numerical

['number_of_courses_viewed',
 'annual_income',
 'interaction_count',
 'lead_score']

In [167]:
for i in numerical:
    df[i] = df[i].fillna(0.0)

In [168]:
# q1 
df['industry'].mode()[0]

'retail'

In [169]:
df[numerical].corrwith(df.converted)

number_of_courses_viewed    0.435914
annual_income               0.053131
interaction_count           0.374573
lead_score                  0.193673
dtype: float64

In [170]:
a = np.zeros((2,2))
a[1,:] = [1,2]
a

array([[0., 0.],
       [1., 2.]])

### correaltion matrix

In [171]:
def get_corr_mat(df, numerical):
    corr_matrix = np.zeros((len(numerical), len(numerical)))
    for i in range(len(numerical)):
        col = numerical[i]
        # print(f"Correlation of all columns with '{col}'")
        corr_matrix[i,:] = corr = df[numerical].corrwith(df[col])
        # print(corr)
    return corr_matrix
    

In [172]:
corr_matrix = get_corr_mat(df, numerical)
corr_matrix

array([[ 1.        ,  0.00977029, -0.02356522, -0.004879  ],
       [ 0.00977029,  1.        ,  0.02703647,  0.01560955],
       [-0.02356522,  0.02703647,  1.        ,  0.00988818],
       [-0.004879  ,  0.01560955,  0.00988818,  1.        ]])

In [173]:
df[numerical].corr()

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.000000,0.009770,-0.023565,-0.004879
annual_income,0.009770,1.000000,0.027036,0.015610
interaction_count,-0.023565,0.027036,1.000000,0.009888
lead_score,-0.004879,0.015610,0.009888,1.000000


In [174]:
# q2 
# max(a,b,c,d) --> d (annual income and interaction_count)

### split dataset

In [175]:
# q3 split dataset
from sklearn.model_selection import train_test_split

In [176]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

In [177]:
len(df_test), len(df_full_train)

(293, 1169)

In [178]:
df_train , df_val = train_test_split(df_full_train , test_size=0.25, random_state=42)

In [179]:
len(df_train), len(df_val)

(876, 293)

In [180]:
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

In [181]:
y_train = df_train.converted.values
y_test = df_test.converted.values
y_val = df_val.converted.values

In [182]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

### mutual info

In [183]:
# q3 
from sklearn.metrics import mutual_info_score

In [184]:
def mutual_score_converted(series):
    return mutual_info_score(series, df_full_train.converted)

In [185]:
mi = df_full_train[categorical_columns].apply(mutual_score_converted)
mi.sort_values(ascending=False)

lead_source          0.025665
employment_status    0.013258
industry             0.011685
location             0.002253
dtype: float64

### hot encoding

In [188]:
# q4
from sklearn.feature_extraction import DictVectorizer

In [189]:
dv = DictVectorizer(sparse=False)

In [190]:
train_dicts = df_train[categorical_columns + numerical].to_dict(orient='records')

In [191]:
X_train = dv.fit_transform(train_dicts)

In [192]:
val_dicts = df_val[categorical_columns + numerical].to_dict(orient='records')

In [193]:
X_val = dv.fit_transform(val_dicts)

In [205]:
test_dicts = df_test[categorical_columns + numerical].to_dict(orient='records')

In [207]:
X_test = dv.fit_transform(test_dicts)

### training with logistic regression

In [208]:
from sklearn.linear_model import LogisticRegression

In [209]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [210]:
model.intercept_[0]

np.float64(-0.0691472802783609)

In [211]:
model.coef_[0].round(3)

array([-0.   , -0.015,  0.034,  0.003,  0.012, -0.103, -0.025,  0.049,
       -0.02 , -0.013, -0.003, -0.009, -0.032, -0.016,  0.311,  0.051,
        0.02 , -0.012, -0.012, -0.115,  0.08 , -0.03 ,  0.004, -0.011,
       -0.011, -0.006,  0.008,  0.006, -0.033, -0.025,  0.454])

In [212]:
y_pred = model.predict_proba(X_val)[:,1]

In [213]:
converted_decision = (y_pred >= 0.5)

In [214]:
(y_val == converted_decision).mean().round(2)

np.float64(0.7)

In [215]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = converted_decision.astype(int)
df_pred['actual'] = y_val

In [216]:
df_pred['correct'] = df_pred.prediction == df_pred.actual

In [217]:
df_pred.correct.mean() # accuracy --> 0.7

np.float64(0.6996587030716723)

In [218]:
# converted_decision.astype(int)

### train with less feature

In [219]:
original_accuracy = df_pred.correct.mean()
original_accuracy

np.float64(0.6996587030716723)

In [220]:
model.fit(X_train , y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [227]:
X_train = df_train[categorical_columns + numerical]
X_val = df_val[categorical_columns + numerical]
X_test = df_test[categorical_columns + numerical]

In [228]:
feature_diffs = {}

In [229]:
for feature in df_train.columns:
    
    X_train_excluded = X_train.drop(columns=[feature])
    X_val_excluded = X_val.drop(columns=[feature])
    X_test_excluded = X_test.drop(columns=[feature])

    X_train_excluded = dv.fit_transform(X_train_excluded.to_dict(orient='records'))
    X_val_excluded = dv.fit_transform(X_val_excluded.to_dict(orient='records'))
    X_test_excluded = dv.fit_transform(X_test_excluded.to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_excluded, y_train)

    y_pred_excluded = model.predict_proba(X_val_excluded)[:,1]

    converted_decision = (y_pred_excluded >= 0.5)

    (y_val == converted_decision).mean()

    df_pred = pd.DataFrame()
    df_pred['probability'] = y_pred_excluded
    df_pred['prediction'] = converted_decision.astype(int)
    df_pred['actual'] = y_val
    
    df_pred['correct'] = df_pred.prediction == df_pred.actual

    new_accuracy = df_pred.correct.mean()

    diff = abs(original_accuracy - new_accuracy)

    feature_diffs[feature] = diff

    print(f"Without '{feature}': accuracy = {new_accuracy:.4f}, diff = {diff:.4f}")

least_useful_feature = min(feature_diffs, key=feature_diffs.get)
print("\nLeast useful feature:", least_useful_feature)
print("Accuracy drop when removed:", feature_diffs[least_useful_feature])

Without 'lead_source': accuracy = 0.7031, diff = 0.0034
Without 'industry': accuracy = 0.6997, diff = 0.0000
Without 'number_of_courses_viewed': accuracy = 0.5563, diff = 0.1433
Without 'annual_income': accuracy = 0.8532, diff = 0.1536
Without 'employment_status': accuracy = 0.6962, diff = 0.0034
Without 'location': accuracy = 0.7099, diff = 0.0102
Without 'interaction_count': accuracy = 0.5563, diff = 0.1433
Without 'lead_score': accuracy = 0.7065, diff = 0.0068

Least useful feature: industry
Accuracy drop when removed: 0.0


In [235]:
# q6
C = [0.01, 0.1, 1, 10, 100]

X_train = df_train[categorical_columns + numerical]
X_val = df_val[categorical_columns + numerical]
X_test = df_test[categorical_columns + numerical]

X_train = dv.fit_transform(X_train.to_dict(orient='records'))
X_val = dv.fit_transform(X_val.to_dict(orient='records'))
X_test  = dv.fit_transform(X_test.to_dict(orient='records'))

for c in C:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred_new = model.predict_proba(X_val)[:,1]

    converted_decision = (y_pred_new >= 0.5)

    (y_val == converted_decision).mean()

    df_pred = pd.DataFrame()
    df_pred['probability'] = y_pred_new
    df_pred['prediction'] = converted_decision.astype(int)
    df_pred['actual'] = y_val
    
    df_pred['correct'] = df_pred.prediction == df_pred.actual

    new_accuracy = df_pred.correct.mean()

    print(f"for C = {c}   , Accuracy = {new_accuracy}")

    
    


for C = 0.01   , Accuracy = 0.6996587030716723
for C = 0.1   , Accuracy = 0.6996587030716723
for C = 1   , Accuracy = 0.6996587030716723
for C = 10   , Accuracy = 0.6996587030716723
for C = 100   , Accuracy = 0.6996587030716723
